In [4]:
import numpy as np
import pandas as pd
import os
print(os.getcwd())

/home/yli/MR_CDFT/MRCDFT_2BCorr


In [32]:
A=136
Nuclei='Xe'
work_dir = '/home/yli/MR_CDFT/MRCDFT_2BCorr/MRCDFT'
para_path = os.path.join(work_dir,'examples/136Xe/136Xe_para.dat')
b23_path = os.path.join(work_dir,'examples/136Xe/136Xe_b23.dat')
output_dir = os.path.join(work_dir,'examples/136Xe/output')



In [33]:
# extract eMax, nbeta, nphi from parameter file
with open(para_path, "r", encoding="utf-8") as f:
    for line in f:
        line_nocomment = line.split("!")[0]

        if "n0f" in line_nocomment:
            eMax = int(line_nocomment.split("=")[1].split()[0])
        if "nphi" in line_nocomment:
            nphi = int(line_nocomment.split("=")[1])
        if "nbeta" in line_nocomment:
            nbeta = int(line_nocomment.split("=")[1])
        if "AMP" in line_nocomment:
            AMP = int(line_nocomment.split("=")[1])
        if "PNP" in line_nocomment:
            PNP = int(line_nocomment.split("=")[1])
        if "Kernels" in line_nocomment:
            Kernels = int(line_nocomment.split("=")[1])
    if(AMP == 0): nbeta = 1
    if(PNP == 0): nphi = 1

# eMax = 6; nphi = 1; nbeta = 1; AMP = 0; Kernels = 2

print('eMax=',eMax,' nphi=',nphi,' nbeta=',nbeta,"AMP=",AMP,"PNP=",PNP," Kernels=",Kernels)

eMax= 6  nphi= 7  nbeta= 12 AMP= 1 PNP= 1  Kernels= 2


In [34]:
oneB_ME_path = os.path.join(output_dir,f'mScheme_1B_A{A}_eMax{eMax:02d}.me')
oneB_ME = pd.read_csv(oneB_ME_path,sep=r'\s+')
oneB_ME = oneB_ME.drop(columns=["n1", "n2","l1", "l2","2j1", "2j2","2j_m1", "2j_m2"])
print(oneB_ME_path)
oneB_ME

/home/yli/MR_CDFT/MRCDFT_2BCorr/MRCDFT/examples/136Xe/output/mScheme_1B_A136_eMax06.me


,ifg,m1,m2,r^2,r^4,r^2Y20,r^4Y20,r^4Y40,f2,Eps1B
0,1,1,1,7.80190,101.44929,-0.00000,-0.00000,-0.0,-0.00000,40.36539
1,1,1,2,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
2,1,1,3,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
3,1,1,4,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
4,1,1,5,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
...,...,...,...,...,...,...,...,...,...,...
85819,2,240,236,0.00000,0.00000,-0.00000,-0.00000,0.0,-0.00000,0.00000
85820,2,240,237,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000
85821,2,240,238,0.00000,0.00000,-0.00000,-0.00000,-0.0,-7.88774,0.00000
85822,2,240,239,0.00000,0.00000,-0.00000,-0.00000,-0.0,-0.00000,0.00000


In [35]:
# Prepare result dataframe, add TD density filename and Kernel filenames

DensList = pd.DataFrame(columns=['beta2(qf)','beta3(qf)', 'beta2(qi)','beta3(qi)', 'Ddens_filename', 'Kernel_filename'])

b23_data = pd.read_csv(b23_path,
                    sep=r'\s+',   # 按空格分隔
                    skiprows=0)
for q1 in range(len(b23_data)):
    for q2 in range(len(b23_data)):
        if(Kernels == 2 and q1 != q2): continue
        beta2q1 = b23_data.iloc[q1]['beta2']
        beta3q1 = b23_data.iloc[q1]['beta3']
        beta2q2 = b23_data.iloc[q2]['beta2']
        beta3q2 = b23_data.iloc[q2]['beta3']

        TDdens_filename = f"Proj_{A}{Nuclei}_D1B.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.me"
        if not os.path.exists(os.path.join(output_dir,TDdens_filename)):
            print(TDdens_filename," not exist !")
        if(q1 <= q2):
            Kernel_filename = f"Proj_{A}{Nuclei}_kern.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}_{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}.elem"
        else:
            Kernel_filename = f"Proj_{A}{Nuclei}_kern.{AMP}D_eMax{eMax:02d}.{nphi:02d}.{nbeta:02d}{int(beta2q2*100):+04d}{int(beta3q2*100):+04d}_{int(beta2q1*100):+04d}{int(beta3q1*100):+04d}.elem"
        if not os.path.exists(os.path.join(output_dir,Kernel_filename)):
            print(Kernel_filename," not exist !")
        
        DensList.loc[len(DensList)] = [beta2q1, beta3q1, beta2q2, beta3q2, TDdens_filename, Kernel_filename]
DensList

,beta2(qf),beta3(qf),beta2(qi),beta3(qi),Ddens_filename,Kernel_filename
0,-0.3,0.0,-0.3,0.0,Proj_136Xe_D1B.1D_eMax06.07.12-030+000_-030+00...,Proj_136Xe_kern.1D_eMax06.07.12-030+000_-030+0...
1,-0.1,0.0,-0.1,0.0,Proj_136Xe_D1B.1D_eMax06.07.12-010+000_-010+00...,Proj_136Xe_kern.1D_eMax06.07.12-010+000_-010+0...
2,0.0,0.0,0.0,0.0,Proj_136Xe_D1B.1D_eMax06.07.12+000+000_+000+00...,Proj_136Xe_kern.1D_eMax06.07.12+000+000_+000+0...
3,0.1,0.0,0.1,0.0,Proj_136Xe_D1B.1D_eMax06.07.12+010+000_+010+00...,Proj_136Xe_kern.1D_eMax06.07.12+010+000_+010+0...
4,0.2,0.0,0.2,0.0,Proj_136Xe_D1B.1D_eMax06.07.12+020+000_+020+00...,Proj_136Xe_kern.1D_eMax06.07.12+020+000_+020+0...
5,0.3,0.0,0.3,0.0,Proj_136Xe_D1B.1D_eMax06.07.12+030+000_+030+00...,Proj_136Xe_kern.1D_eMax06.07.12+030+000_+030+0...
6,0.4,0.0,0.4,0.0,Proj_136Xe_D1B.1D_eMax06.07.12+040+000_+040+00...,Proj_136Xe_kern.1D_eMax06.07.12+040+000_+040+0...
7,0.5,0.0,0.5,0.0,Proj_136Xe_D1B.1D_eMax06.07.12+050+000_+050+00...,Proj_136Xe_kern.1D_eMax06.07.12+050+000_+050+0...
8,0.6,0.0,0.6,0.0,Proj_136Xe_D1B.1D_eMax06.07.12+060+000_+060+00...,Proj_136Xe_kern.1D_eMax06.07.12+060+000_+060+0...
9,0.7,0.0,0.7,0.0,Proj_136Xe_D1B.1D_eMax06.07.12+070+000_+070+00...,Proj_136Xe_kern.1D_eMax06.07.12+070+000_+070+0...


In [36]:
Res_data = DensList.copy()
for q1q2 in range(len(Res_data)):
    Ddens_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Ddens_filename'])
    Kernel_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Kernel_filename'])
    # read D1B density
    Ddens = pd.read_csv(Ddens_path, sep=r'\s+')

    J = 0; Pi = '+'; K1 = 0; K2 = 0
    df_tmp = Ddens[(Ddens["J"]==J) & (Ddens["Pi"]==Pi) & (Ddens["K1"]==K1) & (Ddens["K2"]==K2)]
    merged = pd.merge(df_tmp, oneB_ME, left_on=["ifg","m1", "m2"], right_on=["ifg","m2", "m1"], how="left")
    merged = merged.drop(columns=['m1_y','m2_y']).rename(columns={'m1_x':'m1','m2_x':'m2'})

    # read norm kernel
    with open(Kernel_path, 'r') as f:
        line = f.readlines()[J*4+1]
        Norm = float(line[:15].strip())
        Energy = float(line[30:45].strip())
        # print('Norm=',Norm)
    # print('J=',Ji)

    Res_data.loc[q1q2,f'Norm({J}{Pi})'] = Norm
    Res_data.loc[q1q2,f'Energy({J}{Pi})'] = Energy

    # check particle number
    df_merge_N = merged[merged['m1']==merged['m2']]
    total_neutron = df_merge_N["neutron"].sum()
    total_proton = df_merge_N["proton"].sum()
    Res_data.loc[q1q2,f'N'] = total_neutron/ Norm
    Res_data.loc[q1q2,f'Z'] = total_proton/ Norm

    # calculate r^2
    total_neutron = (merged["neutron"] * merged["r^2"]).sum()
    total_proton = (merged["proton"] * merged["r^2"]).sum()
    Res_data.loc[q1q2,f'r2(n)'] = total_neutron/Norm
    Res_data.loc[q1q2,f'r2(p)'] = total_proton/Norm
    Res_data.loc[q1q2,f'r2(total)'] = total_neutron/Norm + total_proton/Norm
    
    # calculate 1B kernel of eccentricity operator
    total_neutron = (merged["neutron"] * merged["Eps1B"]).sum()
    total_proton = (merged["proton"] * merged["Eps1B"]).sum()
    Res_data.loc[q1q2,f'Eps1B(n)'] = total_neutron/Norm
    Res_data.loc[q1q2,f'Eps1B(p)'] = total_proton/Norm
    Res_data.loc[q1q2,f'Eps1B(total)'] = total_neutron/Norm + total_proton/Norm

Res_data.drop(columns=['Ddens_filename', 'Kernel_filename'])


,beta2(qf),beta3(qf),beta2(qi),beta3(qi),Norm(0+),Energy(0+),N,Z,r2(n),r2(p),r2(total),Eps1B(n),Eps1B(p),Eps1B(total)
0,-0.3,0.0,-0.3,0.0,0.002366,-1103.67520,81.999837,54.000000,2137.767437,1283.588528,3421.355965,25519.869821,15955.464702,41475.334523
1,-0.1,0.0,-0.1,0.0,0.068131,-1121.34160,82.000000,54.000000,2090.248429,1228.593181,3318.841610,25045.159322,14581.979352,39627.138674
2,0.0,0.0,0.0,0.0,0.346700,-1121.03830,82.000000,54.000001,2090.454529,1225.694301,3316.148830,25011.281528,14446.534627,39457.816155
3,0.1,0.0,0.1,0.0,0.078314,-1120.64380,82.000000,54.000000,2090.204837,1228.767572,3318.972409,25018.223544,14578.051650,39596.275193
4,0.2,0.0,0.2,0.0,0.009971,-1113.86090,82.000000,54.000000,2097.262350,1241.311932,3338.574282,25276.012961,14946.546035,40222.558996
5,0.3,0.0,0.3,0.0,0.004826,-1102.29810,81.999679,54.000000,2117.887156,1272.858644,3390.745801,25482.584439,15806.692646,41289.277085
6,0.4,0.0,0.4,0.0,0.003393,-1085.89670,82.000687,54.000000,2143.907507,1304.042497,3447.950004,25416.667816,16296.491516,41713.159332
7,0.5,0.0,0.5,0.0,0.001183,-1057.93390,82.000166,54.000001,2158.361019,1331.910568,3490.271587,25210.779977,16456.027005,41666.806982
8,0.6,0.0,0.6,0.0,0.001522,-1011.99490,82.000001,54.000001,2171.009336,1364.162074,3535.171410,25056.743593,16729.137039,41785.880631
9,0.7,0.0,0.7,0.0,0.001364,-919.50261,82.000003,54.000002,2234.123137,1454.425676,3688.548813,25922.935004,18237.640585,44160.575589


In [37]:
import math
Res_data = DensList.copy()
for q1q2 in range(len(Res_data)):
    Ddens_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Ddens_filename'])
    Kernel_path = os.path.join(output_dir,Res_data.iloc[q1q2]['Kernel_filename'])
    # read D1B density
    Ddens = pd.read_csv(Ddens_path, sep=r'\s+')

    J = 0; Pi = '+'; K1 = 0; K2 = 0
    df_tmp = Ddens[(Ddens["J"]==J) & (Ddens["Pi"]==Pi) & (Ddens["K1"]==K1) & (Ddens["K2"]==K2)]
    merged = pd.merge(df_tmp, oneB_ME, left_on=["ifg","m1", "m2"], right_on=["ifg","m2", "m1"], how="left")
    merged = merged.drop(columns=['m1_y','m2_y']).rename(columns={'m1_x':'m1','m2_x':'m2'})

    # read norm kernel
    with open(Kernel_path, 'r') as f:
        line = f.readlines()[J*4+1]
        Norm = float(line[:15].strip())
        Energy = float(line[30:45].strip())
        # print('Norm=',Norm)
    # print('J=',Ji)

    # calculate r^2
    total_neutron = (merged["neutron"] * merged["r^2"]).sum()
    total_proton = (merged["proton"] * merged["r^2"]).sum()
    Res_data.loc[q1q2,f'r2(total)'] = total_neutron/Norm + total_proton/Norm

    # # calculate r^4
    # total_neutron = (merged["neutron"] * merged["r^4"]).sum()
    # total_proton = (merged["proton"] * merged["r^4"]).sum()
    # Res_data.loc[q1q2,f'r4(total)'] = total_neutron/Norm + total_proton/Norm

    # calculate r^2 Y20
    total_neutron = (merged["neutron"] * merged["r^2Y20"]).sum()
    total_proton = (merged["proton"] * merged["r^2Y20"]).sum()
    Res_data.loc[q1q2,f'r2Y20(total)'] = total_neutron/Norm + total_proton/Norm

    # calculate r^4 Y20
    total_neutron = (merged["neutron"] * merged["r^4Y20"]).sum()
    total_proton = (merged["proton"] * merged["r^4Y20"]).sum()
    Res_data.loc[q1q2,f'r4Y20(total)'] = total_neutron/Norm + total_proton/Norm

    # calculate r^4 Y40
    total_neutron = (merged["neutron"] * merged["r^4Y40"]).sum()
    total_proton = (merged["proton"] * merged["r^4Y40"]).sum()
    Res_data.loc[q1q2,f'r4Y40(total)'] = total_neutron/Norm + total_proton/Norm
    
    # calculate r^2_perp = 2/3 r^2 - 4/3*sqrt(pi/5)r^2 Y20
    Res_data['r^2_perp'] = 2.0/3.0 * Res_data['r2(total)'] - 4.0/3.0*math.sqrt(math.pi/5.0)*Res_data['r2Y20(total)'] 
    # calculate r^4_perp = 8/15 r^4 - 32/21*sqrt(pi/5)r^4 Y20 + 16/105*sqrt(pi)*r^4 Y40
    # Res_data['r^4_perp'] = 8.0/15.0 * Res_data['r4(total)'] - 32.0/21.0*math.sqrt(math.pi/5.0)*Res_data['r4Y20(total)'] + 16.0/105.0*math.sqrt(math.pi)*Res_data['r4Y40(total)']
     
Res_data.drop(columns=['Ddens_filename', 'Kernel_filename'])


,beta2(qf),beta3(qf),beta2(qi),beta3(qi),r2(total),r2Y20(total),r4Y20(total),r4Y40(total),r^2_perp
0,-0.3,0.0,-0.3,0.0,3421.355965,-363.469464,-15103.005437,1674.572201,2665.050230
1,-0.1,0.0,-0.1,0.0,3318.841610,-98.226641,-4282.391964,472.058220,2316.375560
2,0.0,0.0,0.0,0.0,3316.148830,-0.000618,0.088918,-1.899490,2210.766539
3,0.1,0.0,0.1,0.0,3318.972409,101.365752,4506.414318,521.583331,2105.516099
4,0.2,0.0,0.2,0.0,3338.574282,233.796233,10472.207810,793.145304,1978.619924
5,0.3,0.0,0.3,0.0,3390.745801,362.663521,16005.564342,-443.881847,1877.202738
6,0.4,0.0,0.4,0.0,3447.950004,488.192383,21560.893393,-1260.424304,1782.669016
7,0.5,0.0,0.5,0.0,3490.271587,614.340403,27221.968618,-821.063061,1677.559168
8,0.6,0.0,0.6,0.0,3535.171410,739.096166,32717.989471,-820.414051,1575.639604
9,0.7,0.0,0.7,0.0,3688.548813,865.490664,37935.299927,-2219.766439,1544.306469


In [38]:
print(Res_data['r2(total)'])

0    3421.355965
1    3318.841610
2    3316.148830
3    3318.972409
4    3338.574282
5    3390.745801
6    3447.950004
7    3490.271587
8    3535.171410
9    3688.548813
Name: r2(total), dtype: float64
